<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **Space X Falcon 9 First Stage Landing Prediction**

## Lab 2: Data Wrangling

Estimated time needed: **60** minutes


In this lab, we will perform Exploratory Data Analysis (EDA) to find patterns in the data and determine the label for training supervised models.

In the dataset, there are several cases where the booster did not land successfully:
- `True Ocean` — successfully landed in ocean
- `False Ocean` — unsuccessfully landed in ocean
- `True RTLS` — successfully landed on ground pad
- `False RTLS` — unsuccessfully landed on ground pad
- `True ASDS` — successfully landed on drone ship
- `False ASDS` — unsuccessfully landed on drone ship
- `None ASDS` / `None None` — no landing attempted

## Objectives
- Exploratory Data Analysis
- Determine Training Labels (Class column: 1 = success, 0 = failure)


In [1]:
!pip install pandas
!pip install numpy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

### Data Analysis

Load the SpaceX dataset from the previous section.


In [3]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


In [4]:
# Identify and calculate the percentage of missing values in each attribute
missing_pct = df.isnull().sum() / len(df) * 100
print("=== Missing values (%) ===")
print(missing_pct)

=== Missing values (%) ===
FlightNumber       0.000000
Date               0.000000
BoosterVersion     0.000000
PayloadMass        0.000000
Orbit              0.000000
LaunchSite         0.000000
Outcome            0.000000
Flights            0.000000
GridFins           0.000000
Reused             0.000000
Legs               0.000000
LandingPad        28.888889
Block              0.000000
ReusedCount        0.000000
Serial             0.000000
Longitude          0.000000
Latitude           0.000000
dtype: float64


In [5]:
# Identify which columns are numerical and categorical
print("=== Column Data Types ===")
print(df.dtypes)

=== Column Data Types ===
FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object


### TASK 1: Calculate the number of launches on each site

The data contains several SpaceX launch facilities:
- Cape Canaveral Space Launch Complex 40 **(CCAFS LC-40)**
- Vandenberg Air Force Base Space Launch Complex 4E **(VAFB SLC-4E)**
- Kennedy Space Center Launch Complex 39A **(KSC LC-39A)**


In [6]:
# Apply value_counts() on column LaunchSite
launch_site_counts = df['LaunchSite'].value_counts()
print("=== Number of launches per site ===")
print(launch_site_counts)

=== Number of launches per site ===
LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64


### TASK 2: Calculate the number and occurrence of each orbit

Common orbit types in this dataset:
- **LEO** — Low Earth Orbit (<2,000 km)
- **VLEO** — Very Low Earth Orbit (<450 km)
- **GTO** — Geostationary Transfer Orbit
- **ISS** — International Space Station orbit
- **SSO** — Sun-Synchronous Orbit
- **MEO** — Medium Earth Orbit
- **HEO** — High Earth Orbit
- **SO** — Sub-Orbital
- **GEO** — Geostationary Orbit
- **PO** — Polar Orbit
- **ES-L1** — Earth-Sun Lagrange Point 1


In [7]:
# Apply value_counts() on Orbit column
orbit_counts = df['Orbit'].value_counts()
print("=== Number and occurrence of each orbit ===")
print(orbit_counts)

=== Number and occurrence of each orbit ===
Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64


### TASK 3: Calculate the number and occurrence of mission outcome of the orbits


In [8]:
# Apply value_counts() on column Outcome to determine the number of landing outcomes
landing_outcomes = df['Outcome'].value_counts()
print("=== Landing outcomes ===")
print(landing_outcomes)

=== Landing outcomes ===
Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64


In [9]:
# Enumerate all unique landing outcomes
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [10]:
# Create a set of outcomes where the second stage did NOT land successfully
bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
print("Bad outcomes (landing failures):")
print(bad_outcomes)

Bad outcomes (landing failures):
{'False Ocean', 'None ASDS', 'False ASDS', 'None None', 'False RTLS'}


### TASK 4: Create a landing outcome label from Outcome column

Using the `Outcome` column, create a list where:
- **0** = the corresponding row in `Outcome` is in `bad_outcomes` (landing failed)
- **1** = otherwise (landing succeeded)

Assign it to the variable `landing_class`.


In [11]:
# landing_class = 0 if bad_outcome, 1 otherwise
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
print(f"Total records: {len(landing_class)}")
print(f"Successful landings (Class=1): {sum(landing_class)}")
print(f"Failed landings    (Class=0): {len(landing_class) - sum(landing_class)}")

Total records: 90
Successful landings (Class=1): 60
Failed landings    (Class=0): 30


In [12]:
# Add Class column to dataframe
df['Class'] = landing_class
df[['Class']].head(8)

,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,1
7,1


In [13]:
df.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


In [14]:
# Calculate the overall success rate
success_rate = df['Class'].mean()
print(f"Overall landing success rate: {success_rate:.4f} ({success_rate*100:.2f}%)") 

Overall landing success rate: 0.6667 (66.67%)


In [15]:
# Additional EDA: Success rate by Launch Site
print("=== Success rate by Launch Site ===")
print(df.groupby('LaunchSite')['Class'].mean().round(4))

print("\n=== Success rate by Orbit ===")
print(df.groupby('Orbit')['Class'].mean().round(4).sort_values(ascending=False))

print("\n=== Class distribution ===")
print(df['Class'].value_counts())

=== Success rate by Launch Site ===
LaunchSite
CCAFS SLC 40    0.6000
KSC LC 39A      0.7727
VAFB SLC 4E     0.7692
Name: Class, dtype: float64

=== Success rate by Orbit ===
Orbit
ES-L1    1.0000
GEO      1.0000
HEO      1.0000
SSO      1.0000
VLEO     0.8571
LEO      0.7143
PO       0.6667
MEO      0.6667
ISS      0.6190
GTO      0.5185
SO       0.0000
Name: Class, dtype: float64

=== Class distribution ===
Class
1    60
0    30
Name: count, dtype: int64


In [16]:
# Export to CSV for the next section
df.to_csv('dataset_part_2.csv', index=False)
print('dataset_part_2.csv exported successfully.')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

dataset_part_2.csv exported successfully.
Shape: (90, 18)
Columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude', 'Class']


## Authors

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a>

<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>

**Completed by:** [Hrestak33](https://github.com/Hrestak33)

Copyright © 2021 IBM Corporation. All rights reserved.
